In [22]:
from transformers import AutoImageProcessor, AutoModel
from torch.optim.lr_scheduler import CosineAnnealingLR
from datasets import load_dataset
from torch.utils.data import DataLoader
from matplotlib.axes import Axes
import matplotlib.pyplot as plt
from torch.optim import AdamW
from torch import nn, Tensor
import pandas as pd
import numpy as np
from torchmetrics.classification import MultilabelAveragePrecision
import torch, dotenv, os

dotenv.load_dotenv()
token = os.getenv('HF_TOKEN')

In [3]:
ds = load_dataset('Brambles/Ponies', token=token)

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/55 [00:00<?, ?it/s]

In [4]:
to_predict = (
    'female', 'male',
    'unicorn', 'pegasus', 'earth pony', 'alicorn',
    'simple background', 'monochrome',
    'clothes', 'wings', 'horn', 'chest fluff', 'ear fluff', 'hat', 'jewelry', 'food', 'foal',
    'looking at you', 'smiling', 'open mouth', 'blushing', 'sitting', 'raised hoof', 'eyes closed',
    'twilight sparkle', 'fluttershy', 'rainbow dash', 'pinkie pie', 'rarity', 'applejack'
)

In [5]:
epochs = 5
batch_size = 155
lr = 0.001
weight_decay = 1e-3
temp_alpha = 1.0
mix_alpha = 0.3
gamma = 0.001

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base', token=token)

def transform(batch):
    images = [image.convert('RGB') for image in batch['image']]

    pixel_values = processor(
        images=images,
        return_tensors='pt',
    )["pixel_values"]

    return {
        "pixel_values": pixel_values,
        "tags": torch.tensor(batch["tags"], dtype=torch.float32),
    }

In [7]:
dl_train = DataLoader(
    ds['train'].with_transform(transform),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True,
    shuffle=True,
    drop_last=True
)

dl_test = DataLoader(
    ds['test'].with_transform(transform),
    batch_size=batch_size,
    pin_memory=True,
    num_workers=3,
    persistent_workers=True
)

In [8]:
class Tagger(nn.Module):
    def __init__(self, cls_head: nn.Module):
        super().__init__()
        self.backbone_freeze = True

        self.backbone: nn.Module = AutoModel.from_pretrained('facebook/dinov2-base', token=token)
        self.backbone.requires_grad_(False)
        self.classifier = cls_head

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')

        last_layer = self.classifier[-1]
        nn.init.xavier_uniform_(last_layer.weight, gain=0.1)
        nn.init.zeros_(last_layer.bias)

    def forward(self, x):
        if (self.backbone_freeze):
            self.backbone.eval()

            with torch.no_grad():
                embeddings = self.backbone(x).last_hidden_state[:, 0]
        else:
            embeddings = self.backbone(x).last_hidden_state[:, 0]

        return self.classifier(embeddings)

    def freeze_backbone(self, mode=True):
        self.backbone_freeze = mode

        for layer in self.backbone.encoder.layer[-4:]:
            layer.requires_grad_(not mode)

head = nn.Sequential(
    nn.Linear(768, 256),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(256, 30)
)

In [9]:
model = Tagger(head).to(device)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [10]:
#checkpoint = torch.load('outputs/Best-E4.pt')
#model.load_state_dict(checkpoint['model_state_dict'])

In [11]:
optim = AdamW([
        { 'params': model.backbone.parameters(), 'lr': lr * 0.06 },
        { 'params': model.classifier.parameters(), 'lr': lr },
    ],
    weight_decay=weight_decay
)

scheduler = CosineAnnealingLR(optim, epochs)

In [12]:
def vpu_loss(logits: Tensor, labels: Tensor):
    temp = (temp_alpha * logits.std(0, False)).clamp(0.05, 1.0).detach()

    outputs = (logits / temp).sigmoid()
    log_pos_probs = outputs.clamp_min(1e-10).log() * labels

    num_p_rec = 1 / labels.sum(0)
    num_p_rec = torch.where(num_p_rec.isinf(), torch.zeros_like(num_p_rec), num_p_rec)
    num_u_rec = 1 / labels.shape[0]

    p_c = outputs.mean(0)
    u_loss = (num_u_rec * outputs.sum(0)).clamp_min(1e-10).log()
    p_loss = num_p_rec * log_pos_probs.sum(0)

    class_losses = p_c.pow(gamma).detach() * u_loss - p_loss

    return class_losses.sum()


beta_dist = torch.distributions.Beta(mix_alpha, mix_alpha)

def mixup_loss(x: Tensor, preds: Tensor, y: Tensor, model):
    midpoint1 = int(x.shape[0] / 2)
    # This makes sure all partitions are the same size
    midpoint2 = midpoint1 + (1 if x.shape[0] % 2 == 1 else 0)

    x1, x2 = x[:midpoint1], x[midpoint2:] # logits
    y1, y2 = y[:midpoint1], y[midpoint2:] # labels
    yh1, yh2 = preds[:midpoint1], preds[midpoint2:] # preds

    target1 = y1 + yh1 * (y1 == 0)
    target2 = y2 + yh2 * (y2 == 0)

    idx_perm = torch.randperm(len(x1))

    lam = beta_dist.sample()

    x_mixed = x1[idx_perm]      * lam + x2      * (1 - lam)
    y_mixed = target1[idx_perm] * lam + target2 * (1 - lam)

    log_outputs = model(x_mixed).sigmoid().clamp_min(1e-10).log()

    return (((y_mixed.clamp_min(1e-10).log() - log_outputs).pow(2).sum(dim=0)) * (1 / len(x1))).sum()

In [13]:
def confusion_matrix(preds, labels: np.ndarray, tag_names):
    occurences = labels.sum(0)

    tp = ((preds == 1) & (labels == 1)).sum(0)
    tn = ((preds == 0) & (labels == 0)).sum(0)
    fp = ((preds == 1) & (labels == 0)).sum(0)
    fn = ((preds == 0) & (labels == 1)).sum(0)
    prec = tp / (tp + fp)
    rec = tp / (tp + fn)

    return pd.DataFrame({
        'tag': tag_names,
        'f1': (2 * prec * rec / (prec + rec)),
        'precision': prec,
        'recall': rec,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'count': occurences
    }).sort_values('f1', ascending=False)

In [23]:
f_mAP = MultilabelAveragePrecision(len(to_predict))

def evaluate(epoch, prev_best, save_best=True):
    model.eval()
    preds = torch.zeros((len(ds['test']), len(to_predict)))
    test_loss = 0

    with torch.no_grad():
        for i, batch in enumerate(dl_test):
            X: torch.Tensor = batch['pixel_values'].to(device, non_blocking=True)
            y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

            logits: torch.Tensor = model(X)
            test_loss += vpu_loss(logits, y)

            preds[i * batch_size:(i + 1) * batch_size] = logits.sigmoid()

        mAP: Tensor = f_mAP(preds.sigmoid(), ds['test']['tags'])

        test_loss = test_loss / len(dl_test)

        print(f'TL: {test_loss:.3} mAP: {mAP:.3f}\n')

        if mAP > prev_best:
            if save_best:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optim.state_dict(),
                }, f'Best-E{epoch}.pt')

            return mAP

        return prev_best

In [ ]:
def train_epoch(epoch):
    model.train()
    train_loss = 0

    for i, batch in enumerate(dl_train):
        X: torch.Tensor = batch['pixel_values'].to(device, non_blocking=True)
        y: torch.Tensor = batch['tags'].to(device, non_blocking=True)

        logits: Tensor = model(X)
        preds = logits.sigmoid()

        loss = vpu_loss(logits, y) + mixup_loss(X, preds, y, model)
        loss_delta = loss.item()
        train_loss += loss_delta

        loss.backward()
        optim.step()
        scheduler.step()
        optim.zero_grad()

        if (i % 5 == 0):
            print(f'{' '*40}\r{epoch}/{epochs}: %{i * 100 / len(dl_train):.2f}  {train_loss:.2f}  d{loss_delta:.2f}  Avg conf: {preds.sum() * 100 / preds.numel():.2f}%', end='\r')

    print(f'\nAverage Loss: {train_loss / len(dl_train):.3f}{' '*50}')
    return evaluate(epoch, best_mAP)

In [16]:
best_mAP = 0

for epoch in range(1, epochs + 1):
    model.freeze_backbone(epoch == 1)
    best_mAP = train_epoch(epoch)


1/5: %99.522 -12237.43 d-13.23 Avg conf: 38.92%
Average Loss: -11.765                                                  
TL: -21.6 F1: 0.51 Pr: 0.45 Re: 0.73

2/5: %99.522 -26458.35 d-34.61 Avg conf: 27.52%
Average Loss: -25.427                                                  
TL: -40.5 F1: 0.70 Pr: 0.63 Re: 0.86

3/5: %99.522 -32799.13 d-33.34 Avg conf: 28.03%
Average Loss: -31.508                                                  
TL: -41.4 F1: 0.71 Pr: 0.63 Re: 0.86

4/5: %99.522 -35449.07 d-29.86 Avg conf: 30.94%
Average Loss: -34.046                                                  
TL: -41.8 F1: 0.73 Pr: 0.66 Re: 0.86

5/5: %99.522 -37537.00 d-38.50 Avg conf: 26.31%
Average Loss: -36.054                                                  
TL: -44.5 F1: 0.73 Pr: 0.66 Re: 0.85



In [26]:
ds['test']['tags']

Column([[1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], ...])

In [24]:
evaluate(epoch, 0, False)

AttributeError: 'Column' object has no attribute 'shape'

In [17]:
def visualize(rows: pd.DataFrame, preds: np.ndarray, tag_names, width=2):
    imgs = rows['image'].tolist()

    fig, axis = plt.subplots(1, len(preds), figsize=(width * len(rows), width), constrained_layout=True)
    axis: list[Axes]

    for ax, img in zip(axis, imgs):
        ax.imshow(img)
        ax.axis(False)

    labels = np.stack(rows['tags'].to_numpy())
    idx_sorted = preds.argsort()[:, ::-1]

    label_names = np.array(tag_names)

    preds = [
        '\n'.join([
            f'{label_name + ':  ':>20}{p:.2f}  {label}'
            for p, label_name, label in
            zip(preds[o, idx], label_names[idx], labels[o, idx])
            if p > 0.5
        ])
        for o, idx in enumerate(idx_sorted)
    ]

    plt.show()
    print(f'\n\n'.join(preds))

In [18]:
# from utils import visualize
# import numpy as np

# np.random.seed(8)
# idx_obs = np.random.choice(ds['test'], 4)

# rows = df.loc[idx_obs]

# with torch.no_grad():
#     model.eval()

#     logits = model(rows['image']).to(device)
#     logits = logits.sigmoid()

#     visualize(rows, logits.cpu().numpy(), to_predict)